In [ ]:
# OLD - Não funciona
import pandas as pd
import json
import ipywidgets as widgets
from IPython.display import display
from google.colab import files, auth
from google.colab import drive as gdrive
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from oauth2client.client import GoogleCredentials

# Monta o Google Drive
gdrive.mount('/content/drive')

# Autenticação do PyDrive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# Solicita o ID da pasta de imagens
folder_id = input("📂 Cole o ID da pasta do Google Drive com as imagens (ou 0 para ignorar): ")

# Dicionário de mapeamento imagem ↔ material
image_mapping = {}

if folder_id.strip() != "0":
    try:
        image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff', '.webp')
        file_list = drive.ListFile({'q': f"'{folder_id}' in parents and trashed=false"}).GetList()
        for file in file_list:
            filename = file['title']
            if filename.lower().endswith(image_extensions):
                base_name = '.'.join(filename.split('.')[:-1])  # remove extensão
                file_id = file['id']
                # Converte para formato thumbnail
                link = f"https://drive.google.com/thumbnail?id={file_id}&sz=w800"
                image_mapping[base_name] = link
        print(f"🔗 {len(image_mapping)} imagens mapeadas.")
    except Exception as e:
        print(f"⚠️ Erro ao acessar a pasta do Drive: {e}")
        image_mapping = {}

# Upload do Excel
upload_button = widgets.FileUpload(accept='.xlsx', multiple=False)
display(upload_button)

def process_uploaded_file(change):
    try:
        uploaded_file = next(iter(upload_button.value))
        file_name = uploaded_file
        file_content = upload_button.value[file_name]['content']
        file_path = f"/content/{file_name}"

        with open(file_path, "wb") as f:
            f.write(file_content)

        print(f"✔ Arquivo carregado com sucesso: {file_path}")

        # Lê Excel
        df = pd.read_excel(file_path, usecols=[
            'Centro', 'Material', 'Texto breve de material',
            'Texto Longo', 'Depósito', 'Utilização livre'
        ])
        df = df.where(pd.notnull(df), None)

        # Formata 'Depósito' como string sem decimal se for número, ou mantém texto
        df["Depósito"] = df["Depósito"].apply(
            lambda x: str(int(x)) if pd.notnull(x) and isinstance(x, (int, float)) and x == int(x)
            else str(x) if pd.notnull(x)
            else None
        )

        # Mantém 'Utilização livre' como número float (ou None)
        df["Utilização livre"] = df["Utilização livre"].apply(lambda x: float(x) if pd.notnull(x) else None)

        # Adiciona coluna com thumbnail URL (ou None)
        df["Image URL"] = df["Material"].apply(lambda mat: image_mapping.get(str(mat), None))

        # Converte para JSON
        records = []
        for row in df.to_dict(orient='records'):
            record = {key: val if key == "Utilização livre" else str(val) if val is not None else None for key, val in row.items()}
            records.append(record)

        # Salva JSON
        output_path = f"/content/{file_name.rsplit('.', 1)[0]}.json"
        with open(output_path, 'w', encoding='utf-8') as json_file:
            json.dump(records, json_file, ensure_ascii=False, indent=4)

        print(f"📁 JSON gerado com sucesso: {output_path}")
        files.download(output_path)

    except Exception as e:
        print(f"❌ Erro ao processar o arquivo: {e}")

# Listener de upload
upload_button.observe(process_uploaded_file, names='value')


ModuleNotFoundError: No module named 'pydrive'

In [ ]:
#Funcional

import re
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from google.colab import files, auth
from google.colab import drive as gdrive
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.client import GoogleCredentials

# --- Helper: accept folder ID or full URL ---
def extract_folder_id(s: str) -> str:
    m = re.search(r'/folders/([a-zA-Z0-9_-]+)', s)
    return m.group(1) if m else s.strip()

# Monta o Google Drive
gdrive.mount('/content/drive')

# Autenticação do PyDrive2
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# Solicita o ID (ou URL) da pasta de imagens
folder_raw = input("📂 Cole o ID **ou** a URL da pasta do Google Drive com as imagens (ou 0 para ignorar): ")
folder_id = extract_folder_id(folder_raw)

# Dicionário de mapeamento imagem ↔ material
image_mapping = {}

if folder_id != "0":
    try:
        image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff', '.webp')
        file_list = drive.ListFile({'q': f"'{folder_id}' in parents and trashed=false"}).GetList()
        for f in file_list:
            # PyDrive2 can expose either 'title' (v2 style) or 'name' (v3 style). Try both.
            filename = f.get('title') or f.get('name') or ''
            if filename.lower().endswith(image_extensions):
                base_name = '.'.join(filename.split('.')[:-1])  # remove extensão
                file_id = f['id']
                # Link thumbnail rápido do Drive
                link = f"https://drive.google.com/thumbnail?id={file_id}&sz=w800"
                image_mapping[base_name] = link
        print(f"🔗 {len(image_mapping)} imagens mapeadas.")
    except Exception as e:
        print(f"⚠️ Erro ao acessar a pasta do Drive: {e}")
        image_mapping = {}

# Upload do Excel
upload_button = widgets.FileUpload(accept='.xlsx', multiple=False)
display(upload_button)

def process_uploaded_file(change):
    try:
        # Suporte a ipywidgets 7 (dict) e 8 (lista)
        if isinstance(upload_button.value, dict):  # ipywidgets 7
            file_name, meta = next(iter(upload_button.value.items()))
            file_content = meta['content']
        else:  # ipywidgets 8
            meta = upload_button.value[0]
            file_name = meta['name']
            file_content = meta['content']

        file_path = f"/content/{file_name}"
        with open(file_path, "wb") as f:
            f.write(file_content)

        print(f"✔ Arquivo carregado com sucesso: {file_path}")

        # Lê Excel
        df = pd.read_excel(file_path, usecols=[
            'Centro', 'Material', 'Texto breve de material',
            'Texto Longo', 'Depósito', 'Utilização livre'
        ])
        df = df.where(pd.notnull(df), None)

        # 'Depósito' como string sem decimal se for número inteiro
        df["Depósito"] = df["Depósito"].apply(
            lambda x: str(int(x)) if pd.notnull(x) and isinstance(x, (int, float)) and x == int(x)
            else str(x) if pd.notnull(x)
            else None
        )

        # 'Utilização livre' como float (ou None)
        df["Utilização livre"] = df["Utilização livre"].apply(lambda x: float(x) if pd.notnull(x) else None)

        # Adiciona coluna com thumbnail URL (casando pelo nome do arquivo = código do material)
        df["Image URL"] = df["Material"].apply(lambda mat: image_mapping.get(str(mat), None))

        # Converte para JSON preservando número em 'Utilização livre' e texto nos demais
        records = []
        for row in df.to_dict(orient='records'):
            out = {}
            for k, v in row.items():
                if k == "Utilização livre":
                    out[k] = float(v) if v is not None else None
                else:
                    out[k] = str(v) if v is not None else None
            records.append(out)

        out_name = file_name.rsplit('.', 1)[0] + ".json"
        out_path = f"/content/{out_name}"
        with open(out_path, 'w', encoding='utf-8') as json_file:
            json.dump(records, json_file, ensure_ascii=False, indent=4)

        print(f"📁 JSON gerado com sucesso: {out_path}")
        files.download(out_path)

    except Exception as e:
        print(f"❌ Erro ao processar o arquivo: {e}")

# Listener de upload
upload_button.observe(process_uploaded_file, names='value')


Mounted at /content/drive
📂 Cole o ID **ou** a URL da pasta do Google Drive com as imagens (ou 0 para ignorar): 0


FileUpload(value={}, accept='.xlsx', description='Upload')

✔ Arquivo carregado com sucesso: /content/ON01_BASE CADASTRO_2026.02.xlsx
📁 JSON gerado com sucesso: /content/ON01_BASE CADASTRO_2026.02.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>